In [1]:
import numpy as np
import gpflow
import matplotlib.pyplot as plt
import os
import seaborn as sns
import scipy as sp
import pandas as pd
import tensorflow as tf
import ipywidgets as widgets
import pickle
from IPython.display import display
from ipywidgets import interact
from sklearn.preprocessing import StandardScaler


2025-03-24 15:27:16.364025: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-24 15:27:16.364149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-24 15:27:16.935848: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-24 15:27:17.981277: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-24 15:27:31.587070: W tensorflow/compiler/tf2

In [2]:
import numpy as np
import gpflow
import matplotlib.pyplot as plt
import os
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler

# Example config
categories = ['Measured_Normalized']
noise_levels = [0.0]
ns = range(50, 1501, 50)  # sample sizes from 50 to 1500 in increments of 50
seeds = range(10)        # e.g., seed = 0..9

base_dir = '3D-Data'

# ----------------------------------------------------------------------
#  1) Load data into a dictionary keyed by (category, noise_level, n, seed)
# ----------------------------------------------------------------------
train_dict = {}
for category in categories:
    for noise_level in noise_levels:
        for n in ns:
            for seed in seeds:
                file_name = f"rss_n={n}_noise={noise_level}_seed={seed}.csv"
                file_path = os.path.join(base_dir, category, file_name)
                
                if os.path.exists(file_path):
                    data = pd.read_csv(file_path)
                    train_dict[(category, noise_level, n, seed)] = data
                else:
                    print(f"No such file: {file_path}")

# ----------------------------------------------------------------------
#  2) Define a training function that trains GPR models
# ----------------------------------------------------------------------
def gp_trainer(train_dict):
    gp_models = {}
    
    for (category, noise_level, n, seed), df in train_dict.items():
        train_input = df[["RSS0","RSS1","RSS2","RSS3","RSS4"]].values
        train_output = df[["X","Y","Z"]].values
        
        # Example kernel & noise variance
        kernel = gpflow.kernels.SquaredExponential(lengthscales=(0.2,0.2,0.2,0.2,0.2))
        model = gpflow.models.GPR(
            data=(train_input, train_output),
            kernel=kernel,
            noise_variance=1e-4
        )
        
        # Make likelihood trainable, do optimization
        gpflow.set_trainable(model.likelihood, True)
        opt = gpflow.optimizers.Scipy()
        opt.minimize(model.training_loss, model.trainable_variables)
        
        # Store in dictionary
        gp_models[(category, noise_level, n, seed)] = model
        
        print(f"Trained model for: cat={category}, noise={noise_level}, n={n}, seed={seed}")
    
    return gp_models

# ----------------------------------------------------------------------
#  3) Train and save all models
# ----------------------------------------------------------------------
gp_models = gp_trainer(train_dict)

with open('gp_models_absolute.pkl', 'wb') as file:
    pickle.dump(gp_models, file)

print("Finished training all GP models (one per seed).")


2025-03-24 15:29:55.675138: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=0
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=1
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=2
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=3
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=4
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=5
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=6
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=7
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=8
Trained model for: cat=Measured_Normalized, noise=0.0, n=50, seed=9
Trained model for: cat=Measured_Normalized, noise=0.0, n=100, seed=0
Trained model for: cat=Measured_Normalized, noise=0.0, n=100, seed=1
Trained model for: cat=Measured_Normalized, noise=0.0, n=100, seed=2
Trained model for: cat=Measured_Normalized, noise=0.0, n=100, seed=3
Trained model for: cat=Measured_Normalized, 

In [3]:
test_dict={}
m=50
noise_levels= [0.,0.00019,1.9e-05,6e-05, 6e-06]
for category in categories:
    for noise_level in noise_levels:
        file_name= f"gnd_n={m}_noise={noise_level}.csv"
        file_path = os.path.join(base_dir,category,file_name)
        if os.path.exists(file_path):
            test_dict[(category,noise_level)] = pd.read_csv(file_path)
        else:
            print('no such file', category)

In [4]:
with open('gp_models_absolute.pkl', 'rb') as file:
    gp_models = pickle.load(file)

In [4]:
def make_predictions(
    gp_model,
    test_data,
    prediction_save_directory="predictions/gps_absolute",
    n=9,
    noise_level=0.,
    seed=0,
    scaler="scaler_absolute.pkl"
):
    """
    Make predictions for the given GP model and test data, then save to CSV.
    Now includes `seed` in the output filename to avoid overwriting.
    """
    import pickle  # Make sure to import if not already done
    import numpy as np
    import os

    output_columns = ["X", "Y", "Z"]
    
    # Build a filename that includes the seed
    filename = f"rss_n={n}_noise={noise_level}_seed={seed}.csv"
    
    RSS_test = test_data.drop(output_columns, axis=1).values
    mean, covariance = gp_model.predict_f(RSS_test)
    
    with open(scaler, 'rb') as scaler_file:
        loaded_scaler = pickle.load(scaler_file)
    
    # Descale the predictions
    predictions = mean
    descaled = loaded_scaler.inverse_transform(
        np.hstack([np.zeros_like(RSS_test), predictions])
    )[:, -3:]  # keep only the last 3 columns (X, Y, Z)
    
    predictions_df = pd.DataFrame(descaled, columns=output_columns)
    
    # Ensure directory exists
    if not os.path.exists(prediction_save_directory):
        os.makedirs(prediction_save_directory)
    
    # Save predictions
    prediction_save_path = os.path.join(prediction_save_directory, filename)
    predictions_df.to_csv(prediction_save_path, index=False)
    print(f"Saved predictions to {prediction_save_path}")
    
    return predictions_df


In [5]:
# i is (category, noise_level, n, seed)
for i in gp_models.keys():
    for j in test_dict.keys():
        make_predictions(
            gp_models[i],
            test_dict[j],
            n=i[2],
            noise_level=j[1],
            seed=i[3],                # <--- Pass the seed from the key
            prediction_save_directory="predictions/gps_absolute"
        )


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Saved predictions to predictions/gps_absolute/rss_n=50_noise=0.0_seed=0.csv


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Saved predictions to predictions/gps_absolute/rss_n=50_noise=0.00019_seed=0.csv


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


...

Saved predictions to predictions/gps_absolute/rss_n=1500_noise=1.9e-05_seed=9.csv


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Saved predictions to predictions/gps_absolute/rss_n=1500_noise=6e-05_seed=9.csv


/home/sumo/anikraft/miniconda3/envs/trieste_env/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Saved predictions to predictions/gps_absolute/rss_n=1500_noise=6e-06_seed=9.csv
